# Ultralytics YOLOv8 Training Notebook

## This Section is for Training Yolo

In [1]:
import os
import shutil
import yaml
import json
import numpy as np
import glob
import optuna
import torch
from ultralytics import YOLO
import ultralytics.data.build as build
from ultralytics.data.dataset import YOLODataset

/Users/ahmadfariz/Projects/github/MARROWS/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class YOLOWeightedDataset(YOLODataset):
    def __init__(self, *args, mode="train", **kwargs):
        super(YOLOWeightedDataset, self).__init__(*args, **kwargs)
        self.train_mode = "train" in getattr(self, "prefix", "")
        self.count_instances()
        class_weights = np.sum(self.counts) / self.counts
        self.class_weights = np.array(class_weights)
        self.agg_func = np.mean
        self.weights = self.calculate_weights()
        self.probabilities = self.calculate_probabilities()
        print(f"✅ Using YOLOWeightedDataset with {len(self.counts)} classes.")
        print(f"Class counts: {self.counts.tolist()}")
        print(f"Sampling probabilities (first 10): {self.probabilities[:10]}")
    
    def count_instances(self):
        self.counts = [0 for _ in range(len(self.data.get("names", [])))]
        for label in getattr(self, "labels", []):
            cls = label.get('cls', np.array([])).reshape(-1).astype(int)
            for id in cls:
                if 0 <= id < len(self.counts):
                    self.counts[id] += 1
        self.counts = np.array(self.counts)
        self.counts = np.where(self.counts == 0, 1, self.counts)

    def calculate_weights(self):
        weights = []
        for label in getattr(self, "labels", []):
            cls = label.get('cls', np.array([])).reshape(-1).astype(int)
            if cls.size == 0:
                weights.append(1.0)
                continue
            weight = float(self.agg_func(self.class_weights[cls]))
            weights.append(weight)
        return weights

    def calculate_probabilities(self):
        total_weight = float(sum(self.weights)) if len(self.weights) > 0 else 1.0
        if total_weight == 0:
            return [1.0 / len(self.weights)] * len(self.weights) if len(self.weights) > 0 else []
        return [w / total_weight for w in self.weights]

    def __getitem__(self, index):
        if not self.train_mode:
            return self.transforms(self.get_image_and_label(index))
        else:
            if not self.probabilities:
                return self.transforms(self.get_image_and_label(index))
            idx = np.random.choice(len(self.labels), p=self.probabilities)
            return self.transforms(self.get_image_and_label(idx))

build.YOLODataset = YOLOWeightedDataset

#### insert Variable in the section below and you can find the results by opening the path and variable to the projects

In [3]:
dataset = "../../data/dataset_split/data.yaml"
model_path = "yolo11n-seg.pt"
study_name = "YOLO11_Marrows_2.1_tuning"
trial_group = "hyperparameters_tuning"
directory = f"../optuna_study/{study_name}/{trial_group}"
storage_path = "sqlite:///../../results/db/train_parameters.db"

os.makedirs(directory, exist_ok=True)

In [4]:
import os

print(os.path.exists(dataset))  # True kalau file ada
print(os.path.abspath(dataset)) # tampilkan path absolutnya

True
/Users/ahmadfariz/Projects/github/MARROWS/data/dataset_split/data.yaml


In [5]:
def train_model(hyp, trial_num, use_default=False):
    trial_name = f"{trial_group}_{trial_num}"
    project_dir = os.path.join(directory, trial_name)

    if os.path.exists(project_dir):
        shutil.rmtree(project_dir)

    hyp_path = None
    if not use_default:
        hyp_path = f"{trial_name}_hyp.yaml"
        with open(hyp_path, 'w') as f:
            yaml.dump(hyp, f)

    device = "cpu"

    model = YOLO(model_path)

     # Train
    results = model.train(
        data=dataset,
        epochs=150,
        imgsz=640,
        batch=8,
        name=trial_name,
        cfg=hyp_path if hyp_path else None,
        patience=15,  # Early stopping
        device=device,
    )

    try:
        result_dir = project_dir
        best_weight = os.path.join(result_dir, "weights", "best.pt")
        if os.path.exists(best_weight):
            shutil.copy(best_weight, f"{trial_name}_best.pt")

        metrics_json = os.path.join(result_dir, "metrics.json")
        if os.path.exists(metrics_json):
            with open(metrics_json, 'r') as f:
                metrics_dict = json.load(f)
        else:
            metrics_dict = {}

        cm_file = os.path.join(result_dir, "confusion_matrix.png")
        if os.path.exists(cm_file):
            shutil.copy(cm_file, f"{trial_name}_confusion_matrix.png")

        csv_file = os.path.join(result_dir, "results.csv")
        if os.path.exists(csv_file):
            shutil.copy(csv_file, f"{trial_name}_results.csv")

        return metrics_dict.get("metrics/mAP50(B)", 0.0)

    except Exception as e:
        print(f"⚠️ Error saving results for trial {trial_num}: {e}")
        return 0.0

In [6]:
def objective(trial):
    if trial.number == 0:
        print("🚀 Running baseline trial with default YOLOv11 hyperparameters...")
        return train_model(None, trial.number, use_default=True)
    
    hyp = {
    # Non-augmentasi
    "lr0": trial.suggest_float("lr0", 1e-5, 1e-1, log=True),
    "lrf": trial.suggest_float("lrf", 0.01, 1.0),
    "momentum": trial.suggest_float("momentum", 0.6, 0.98),
    "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.001),
    "warmup_epochs": trial.suggest_float("warmup_epochs", 0.0, 5.0),
    "warmup_momentum": trial.suggest_float("warmup_momentum", 0.0, 0.95),
    "box": trial.suggest_float("box", 0.02, 0.2),
    "cls": trial.suggest_float("cls", 0.2, 4.0),

    # Augmentasi aman untuk cell
    "hsv_h": trial.suggest_float("hsv_h", 0.0, 0.02),
    "hsv_s": trial.suggest_float("hsv_s", 0.0, 0.15),
    "hsv_v": trial.suggest_float("hsv_v", 0.0, 0.15),
    "degrees": trial.suggest_float("degrees", 0.0, 5.0),
    "translate": trial.suggest_float("translate", 0.0, 0.05),
    "scale": trial.suggest_float("scale", 0.9, 1.1),
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.0,
    "mosaic": 0.0,
    "mixup": 0.0,
    "copy_paste": 0.0,
}

    try:
        return train_model(hyp, trial.number)
    except Exception as e:
        print(f"⚠️ Trial {trial.number} failed: {e}")
        return 0.0

In [ ]:
if __name__ == "__main__":
    study = optuna.create_study(
        direction="maximize",
        study_name=study_name,
        storage=storage_path,
        load_if_exists=True
    )
    study.optimize(objective, n_trials=50)

    print("✅ Tuning complete!")
    print("Best Trial:")
    print(study.best_trial)

[I 2025-08-20 05:57:52,070] Using an existing study with name 'YOLO11_Marrows_2.1_tuning' instead of creating a new one.


New https://pypi.org/project/ultralytics/8.3.181 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.165 🚀 Python-3.13.3 torch-2.7.1 CPU (Apple M2)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=0.07042304429057038, cache=False, cfg=hyperparameters_tuning_1_hyp.yaml, classes=None, close_mosaic=10, cls=2.0310086177259272, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../data/dataset_split/data.yaml, degrees=0.19440678208865692, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.010952066559367077, hsv_s=0.09227406358070583, hsv_v=0.06269493131965939, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=1.9455324983457025e-05, lrf=0.28414150703856267, mask_ratio=4, max_det=300, mix

2025-08-20 05:58:02,787	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2025-08-20 05:58:04,756	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


Overriding model.yaml nc=80 with nc=17

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultralytics.nn.modules.block.C3k2            [128, 128, 1, True]           
  7                  -1  1    295424  ultralytic

train: Scanning /Users/ahmadfariz/Projects/github/MARROWS/data/dataset_split/labels/train... 341 images, 115 backgrounds, 0 corrupt: 100%|██████████| 456/456 [00:00<00:00, 2476.37it/s]

...

val: New cache created: /Users/ahmadfariz/Projects/github/MARROWS/data/dataset_split/labels/val.cache


✅ Using YOLOWeightedDataset with 17 classes.
Class counts: [34, 58, 17, 29, 10, 23, 22, 27, 43, 11, 26, 20, 16, 26, 21, 123, 9]
Sampling probabilities (first 10): [0.0006893339242446164, 0.0045035226905285886, 0.010552974767295125, 0.03227336099872522, 0.027230648342674402, 0.03227336099872522, 0.01132839707871395, 0.004985315926632351, 0.013220146295810548, 0.011172922469517076]
Plotting labels to runs/segment/hyperparameters_tuning_1/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=1.9455324983457025e-05' and 'momentum=0.8664259922567609' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.000476, momentum=0.9) with parameter groups 90 weight(decay=0.0), 101 weight(decay=0.0005951423670625544), 100 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to runs/segment/hyperparameters_tuning_1
Starting training for 150 epochs...

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_lo

      1/150         0G   0.009358    0.01623      19.74      1.012         33        640: 100%|██████████| 57/57 [02:05<00:00,  2.21s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:10<00:00,  1.83s/it]

                   all         81        515     0.0181     0.0441       0.03     0.0242     0.0181     0.0441       0.03     0.0254

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss  Instances       Size



      2/150         0G   0.009392    0.01294       17.1     0.9757         25        640:  75%|███████▌  | 43/57 [01:35<00:30,  2.17s/it]

In [ ]:
best_trial = study.best_trial
best_trial_number = best_trial.number
best_model_path = f"runs/detect/trial_Hyper_{best_trial.number}/weights/best.pt"

In [ ]:
print(f"Best model saved at: {best_model_path}")

In [ ]:
print(study.trials)